In [2]:
import pandas as pd
import numpy as np

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Pandas: 3.0.3
NumPy: 2.4.4


In [3]:
train = pd.read_csv("fraudTrain.csv")
test = pd.read_csv("fraudTest.csv")

print("Train:", train.shape)
print("Test:", test.shape)

Train: (1296675, 23)
Test: (555719, 23)


In [4]:
train_df = train.copy()
test_df = test.copy()

In [5]:
train_df["trans_date_trans_time"] = pd.to_datetime(
    train_df["trans_date_trans_time"]
)

test_df["trans_date_trans_time"] = pd.to_datetime(
    test_df["trans_date_trans_time"]
)

train_df["dob"] = pd.to_datetime(train_df["dob"])
test_df["dob"] = pd.to_datetime(test_df["dob"])

In [6]:
train_df = train_df.sort_values(
    "trans_date_trans_time"
).reset_index(drop=True)

test_df = test_df.sort_values(
    "trans_date_trans_time"
).reset_index(drop=True)

In [7]:
columns_to_drop = [
    "Unnamed: 0",
    "first",
    "last",
    "street",
    "trans_num"
]

train_df = train_df.drop(columns=columns_to_drop)
test_df = test_df.drop(columns=columns_to_drop)

In [8]:
for df in [train_df, test_df]:
    df["hour"] = df["trans_date_trans_time"].dt.hour
    df["day_of_week"] = df["trans_date_trans_time"].dt.dayofweek
    df["month"] = df["trans_date_trans_time"].dt.month

In [9]:
for df in [train_df, test_df]:
    df["customer_age"] = (
        df["trans_date_trans_time"].dt.year
        - df["dob"].dt.year
    )

In [10]:
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

Train shape: (1296675, 22)
Test shape: (555719, 22)

Columns:
['trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'gender', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud', 'hour', 'day_of_week', 'month', 'customer_age']


In [11]:
print(
    train_df[
        [
            "trans_date_trans_time",
            "dob",
            "hour",
            "day_of_week",
            "month",
            "customer_age"
        ]
    ].head(10)
)

  trans_date_trans_time        dob  hour  day_of_week  month  customer_age
0   2019-01-01 00:00:18 1988-03-09     0            1      1            31
1   2019-01-01 00:00:44 1978-06-21     0            1      1            41
2   2019-01-01 00:00:51 1962-01-19     0            1      1            57
3   2019-01-01 00:01:16 1967-01-12     0            1      1            52
4   2019-01-01 00:03:06 1986-03-28     0            1      1            33
5   2019-01-01 00:04:08 1961-06-19     0            1      1            58
6   2019-01-01 00:04:42 1993-08-16     0            1      1            26
7   2019-01-01 00:05:08 1947-08-21     0            1      1            72
8   2019-01-01 00:05:18 1941-03-07     0            1      1            78
9   2019-01-01 00:06:01 1974-03-28     0            1      1            45


In [12]:
all_df = pd.concat(
    [
        train_df.assign(dataset="train"),
        test_df.assign(dataset="test")
    ],
    ignore_index=True
)

In [13]:
all_df = all_df.sort_values(
    ["cc_num", "trans_date_trans_time"]
).reset_index(drop=True)

In [14]:
print(all_df.shape)
print(all_df["dataset"].value_counts())

(1852394, 23)
dataset
train    1296675
test      555719
Name: count, dtype: int64


In [15]:
all_df["prev_trans_time"] = (
    all_df.groupby("cc_num")["trans_date_trans_time"]
    .shift(1)
)

In [16]:
all_df["time_since_prev_txn"] = (
    all_df["trans_date_trans_time"] - all_df["prev_trans_time"]
).dt.total_seconds() / 60

In [17]:
print(all_df[
    [
        "cc_num",
        "trans_date_trans_time",
        "prev_trans_time",
        "time_since_prev_txn"
    ]
].head(10))

        cc_num trans_date_trans_time     prev_trans_time  time_since_prev_txn
0  60416207185   2019-01-01 12:47:15                 NaT                  NaN
1  60416207185   2019-01-02 08:44:57 2019-01-01 12:47:15          1197.700000
2  60416207185   2019-01-02 08:47:36 2019-01-02 08:44:57             2.650000
3  60416207185   2019-01-02 12:38:14 2019-01-02 08:47:36           230.633333
4  60416207185   2019-01-02 13:10:46 2019-01-02 12:38:14            32.533333
5  60416207185   2019-01-03 13:56:35 2019-01-02 13:10:46          1485.816667
6  60416207185   2019-01-03 17:05:10 2019-01-03 13:56:35           188.583333
7  60416207185   2019-01-04 13:59:55 2019-01-03 17:05:10          1254.750000
8  60416207185   2019-01-04 21:17:22 2019-01-04 13:59:55           437.450000
9  60416207185   2019-01-05 00:42:24 2019-01-04 21:17:22           205.033333


In [18]:
print(all_df["time_since_prev_txn"].describe())

count    1.851395e+06
mean     5.156580e+02
std      7.554832e+02
min      0.000000e+00
25%      9.496667e+01
50%      2.615833e+02
75%      6.355833e+02
max      2.235785e+04
Name: time_since_prev_txn, dtype: float64


In [19]:
print(
    "Missing previous transaction:",
    all_df["time_since_prev_txn"].isna().sum()
)

Missing previous transaction: 999


In [20]:
print(
    all_df[
        all_df["dataset"].eq("test")
    ][
        [
            "cc_num",
            "trans_date_trans_time",
            "prev_trans_time",
            "time_since_prev_txn"
        ]
    ].head(10)
)

           cc_num trans_date_trans_time     prev_trans_time  \
1518  60416207185   2020-06-21 13:05:42 2020-06-21 08:54:21   
1519  60416207185   2020-06-21 16:25:36 2020-06-21 13:05:42   
1520  60416207185   2020-06-22 07:58:33 2020-06-21 16:25:36   
1521  60416207185   2020-06-22 15:32:31 2020-06-22 07:58:33   
1522  60416207185   2020-06-23 12:28:54 2020-06-22 15:32:31   
1523  60416207185   2020-06-23 14:24:48 2020-06-23 12:28:54   
1524  60416207185   2020-06-23 16:39:40 2020-06-23 14:24:48   
1525  60416207185   2020-06-23 19:07:05 2020-06-23 16:39:40   
1526  60416207185   2020-06-23 22:45:57 2020-06-23 19:07:05   
1527  60416207185   2020-06-24 04:22:17 2020-06-23 22:45:57   

      time_since_prev_txn  
1518           251.350000  
1519           199.900000  
1520           932.950000  
1521           453.966667  
1522          1256.383333  
1523           115.900000  
1524           134.866667  
1525           147.416667  
1526           218.866667  
1527           336.333333 

In [21]:
all_df["prev_txn_amt"] = (
    all_df.groupby("cc_num")["amt"]
    .shift(1)
)

In [22]:
print(
    all_df[
        [
            "cc_num",
            "trans_date_trans_time",
            "amt",
            "prev_txn_amt"
        ]
    ].head(5)
)

        cc_num trans_date_trans_time    amt  prev_txn_amt
0  60416207185   2019-01-01 12:47:15   7.27           NaN
1  60416207185   2019-01-02 08:44:57  52.94          7.27
2  60416207185   2019-01-02 08:47:36  82.08         52.94
3  60416207185   2019-01-02 12:38:14  34.79         82.08
4  60416207185   2019-01-02 13:10:46  27.18         34.79


In [23]:
print(
    "Missing previous amount:",
    all_df["prev_txn_amt"].isna().sum()
)

Missing previous amount: 999


In [24]:
all_df["amount_change_ratio"] = (
    all_df["amt"] / all_df["prev_txn_amt"]
)

In [25]:
print(
    all_df[
        [
            "cc_num",
            "trans_date_trans_time",
            "amt",
            "prev_txn_amt",
            "amount_change_ratio"
        ]
    ].head(20)
)

         cc_num trans_date_trans_time     amt  prev_txn_amt  \
0   60416207185   2019-01-01 12:47:15    7.27           NaN   
1   60416207185   2019-01-02 08:44:57   52.94          7.27   
2   60416207185   2019-01-02 08:47:36   82.08         52.94   
3   60416207185   2019-01-02 12:38:14   34.79         82.08   
4   60416207185   2019-01-02 13:10:46   27.18         34.79   
5   60416207185   2019-01-03 13:56:35    6.87         27.18   
6   60416207185   2019-01-03 17:05:10    8.43          6.87   
7   60416207185   2019-01-04 13:59:55  117.11          8.43   
8   60416207185   2019-01-04 21:17:22   26.74        117.11   
9   60416207185   2019-01-05 00:42:24  105.20         26.74   
10  60416207185   2019-01-05 21:34:20    4.98        105.20   
11  60416207185   2019-01-06 10:25:49  102.47          4.98   
12  60416207185   2019-01-07 12:58:19  204.15        102.47   
13  60416207185   2019-01-08 08:05:23   64.31        204.15   
14  60416207185   2019-01-08 23:20:22  200.77         6

In [26]:
print(all_df["amount_change_ratio"].describe())

count    1.851395e+06
mean     5.592703e+00
std      3.083843e+01
min      4.941045e-05
25%      3.215171e-01
50%      1.000579e+00
75%      3.108393e+00
max      1.147779e+04
Name: amount_change_ratio, dtype: float64


In [27]:
all_df["log_amount_change_ratio"] = np.log(
    all_df["amount_change_ratio"]
)

In [28]:
print(
    all_df[
        [
            "amt",
            "prev_txn_amt",
            "amount_change_ratio",
            "log_amount_change_ratio"
        ]
    ].head(10)
)

      amt  prev_txn_amt  amount_change_ratio  log_amount_change_ratio
0    7.27           NaN                  NaN                      NaN
1   52.94          7.27             7.281981                 1.985403
2   82.08         52.94             1.550434                 0.438535
3   34.79         82.08             0.423855                -0.858364
4   27.18         34.79             0.781259                -0.246849
5    6.87         27.18             0.252759                -1.375317
6    8.43          6.87             1.227074                 0.204633
7  117.11          8.43            13.892052                 2.631317
8   26.74        117.11             0.228332                -1.476953
9  105.20         26.74             3.934181                 1.369703


In [29]:
print(all_df["log_amount_change_ratio"].describe())

count    1.851395e+06
mean    -5.896265e-05
std      1.868419e+00
min     -9.915349e+00
25%     -1.134704e+00
50%      5.789829e-04
75%      1.134106e+00
max      9.348169e+00
Name: log_amount_change_ratio, dtype: float64


In [30]:
all_df["prev_avg_amt"] = (
    all_df.groupby("cc_num")["amt"]
    .transform(lambda x: x.expanding().mean().shift(1))
)

In [31]:
print(
    all_df[
        [
            "cc_num",
            "trans_date_trans_time",
            "amt",
            "prev_avg_amt"
        ]
    ].head(10)
)

        cc_num trans_date_trans_time     amt  prev_avg_amt
0  60416207185   2019-01-01 12:47:15    7.27           NaN
1  60416207185   2019-01-02 08:44:57   52.94      7.270000
2  60416207185   2019-01-02 08:47:36   82.08     30.105000
3  60416207185   2019-01-02 12:38:14   34.79     47.430000
4  60416207185   2019-01-02 13:10:46   27.18     44.270000
5  60416207185   2019-01-03 13:56:35    6.87     40.852000
6  60416207185   2019-01-03 17:05:10    8.43     35.188333
7  60416207185   2019-01-04 13:59:55  117.11     31.365714
8  60416207185   2019-01-04 21:17:22   26.74     42.083750
9  60416207185   2019-01-05 00:42:24  105.20     40.378889


In [32]:
print(
    all_df[
        [
            "cc_num",
            "trans_date_trans_time",
            "amt",
            "prev_avg_amt"
        ]
    ].head(10)
)

        cc_num trans_date_trans_time     amt  prev_avg_amt
0  60416207185   2019-01-01 12:47:15    7.27           NaN
1  60416207185   2019-01-02 08:44:57   52.94      7.270000
2  60416207185   2019-01-02 08:47:36   82.08     30.105000
3  60416207185   2019-01-02 12:38:14   34.79     47.430000
4  60416207185   2019-01-02 13:10:46   27.18     44.270000
5  60416207185   2019-01-03 13:56:35    6.87     40.852000
6  60416207185   2019-01-03 17:05:10    8.43     35.188333
7  60416207185   2019-01-04 13:59:55  117.11     31.365714
8  60416207185   2019-01-04 21:17:22   26.74     42.083750
9  60416207185   2019-01-05 00:42:24  105.20     40.378889


In [33]:
sample_card = all_df["cc_num"].iloc[0]

print(
    all_df[
        all_df["cc_num"] == sample_card
    ][
        [
            "trans_date_trans_time",
            "amt",
            "prev_avg_amt"
        ]
    ].head(10)
)

  trans_date_trans_time     amt  prev_avg_amt
0   2019-01-01 12:47:15    7.27           NaN
1   2019-01-02 08:44:57   52.94      7.270000
2   2019-01-02 08:47:36   82.08     30.105000
3   2019-01-02 12:38:14   34.79     47.430000
4   2019-01-02 13:10:46   27.18     44.270000
5   2019-01-03 13:56:35    6.87     40.852000
6   2019-01-03 17:05:10    8.43     35.188333
7   2019-01-04 13:59:55  117.11     31.365714
8   2019-01-04 21:17:22   26.74     42.083750
9   2019-01-05 00:42:24  105.20     40.378889


In [34]:
all_df["amount_vs_prev_avg"] = (
    all_df["amt"] / all_df["prev_avg_amt"]
)

In [35]:
print(
    all_df[
        [
            "amt",
            "prev_avg_amt",
            "amount_vs_prev_avg"
        ]
    ].head(20)
)

       amt  prev_avg_amt  amount_vs_prev_avg
0     7.27           NaN                 NaN
1    52.94      7.270000            7.281981
2    82.08     30.105000            2.726457
3    34.79     47.430000            0.733502
4    27.18     44.270000            0.613960
5     6.87     40.852000            0.168168
6     8.43     35.188333            0.239568
7   117.11     31.365714            3.733695
8    26.74     42.083750            0.635400
9   105.20     40.378889            2.605322
10    4.98     46.861000            0.106272
11  102.47     43.053636            2.380054
12  204.15     48.005000            4.252682
13   64.31     60.016154            1.071545
14  200.77     60.322857            3.328257
15   81.48     69.686000            1.169245
16   76.33     70.423125            1.083877
17   52.47     70.770588            0.741410
18    1.90     69.753889            0.027239
19    7.13     66.182632            0.107732


In [36]:
print(all_df["amount_vs_prev_avg"].describe())

count    1.851395e+06
mean     1.013245e+00
std      2.592547e+00
min      2.176430e-03
25%      1.560888e-01
50%      6.677805e-01
75%      1.198126e+00
max      6.763881e+02
Name: amount_vs_prev_avg, dtype: float64


In [37]:
all_df["txn_count_1h"] = (
    all_df.groupby("cc_num", group_keys=False)
    .apply(
        lambda g: (
            g.set_index("trans_date_trans_time")["amt"]
            .rolling("1h", closed="left")
            .count()
            .to_numpy()
        ),
        include_groups=False
    )
)

In [38]:
all_df["txn_count_24h"] = (
    all_df.groupby("cc_num", group_keys=False)
    .apply(
        lambda g: (
            g.set_index("trans_date_trans_time")["amt"]
            .rolling("24h", closed="left")
            .count()
            .to_numpy()
        ),
        include_groups=False
    )
)

In [39]:
print(
    all_df[
        [
            "cc_num",
            "trans_date_trans_time",
            "txn_count_1h",
            "txn_count_24h"
        ]
    ].head(20)
)

         cc_num trans_date_trans_time txn_count_1h txn_count_24h
0   60416207185   2019-01-01 12:47:15          NaN           NaN
1   60416207185   2019-01-02 08:44:57          NaN           NaN
2   60416207185   2019-01-02 08:47:36          NaN           NaN
3   60416207185   2019-01-02 12:38:14          NaN           NaN
4   60416207185   2019-01-02 13:10:46          NaN           NaN
5   60416207185   2019-01-03 13:56:35          NaN           NaN
6   60416207185   2019-01-03 17:05:10          NaN           NaN
7   60416207185   2019-01-04 13:59:55          NaN           NaN
8   60416207185   2019-01-04 21:17:22          NaN           NaN
9   60416207185   2019-01-05 00:42:24          NaN           NaN
10  60416207185   2019-01-05 21:34:20          NaN           NaN
11  60416207185   2019-01-06 10:25:49          NaN           NaN
12  60416207185   2019-01-07 12:58:19          NaN           NaN
13  60416207185   2019-01-08 08:05:23          NaN           NaN
14  60416207185   2019-01

In [40]:
print(all_df[["txn_count_1h", "txn_count_24h"]].describe())

       txn_count_1h txn_count_24h
count             0             0
unique            0             0
top             NaN           NaN
freq            NaN           NaN


In [41]:
all_df = all_df.drop(
    columns=["txn_count_1h", "txn_count_24h"]
)

In [ ]:
all_df = all_df.sort_values(
    ["cc_num", "trans_date_trans_time"]
).reset_index(drop=True)

In [44]:
def count_previous_transactions(group, window):
    times = group["trans_date_trans_time"].values.astype("datetime64[ns]")
    
    left = times - np.timedelta64(window, "h")
    
    return np.searchsorted(times, times, side="left") - np.searchsorted(
        times, left, side="left"
    )

In [45]:
all_df["txn_count_1h"] = (
    all_df.groupby("cc_num", group_keys=False)
    .apply(
        lambda g: pd.Series(
            count_previous_transactions(g, 1),
            index=g.index
        ),
        include_groups=False
    )
    .reset_index(level=0, drop=True)
)

In [46]:
all_df["txn_count_24h"] = (
    all_df.groupby("cc_num", group_keys=False)
    .apply(
        lambda g: pd.Series(
            count_previous_transactions(g, 24),
            index=g.index
        ),
        include_groups=False
    )
    .reset_index(level=0, drop=True)
)

In [47]:
print(
    all_df[
        [
            "cc_num",
            "trans_date_trans_time",
            "txn_count_1h",
            "txn_count_24h"
        ]
    ].head(10)
)

        cc_num trans_date_trans_time  txn_count_1h  txn_count_24h
0  60416207185   2019-01-01 12:47:15             0              0
1  60416207185   2019-01-02 08:44:57             0              1
2  60416207185   2019-01-02 08:47:36             1              2
3  60416207185   2019-01-02 12:38:14             0              3
4  60416207185   2019-01-02 13:10:46             1              3
5  60416207185   2019-01-03 13:56:35             0              0
6  60416207185   2019-01-03 17:05:10             0              1
7  60416207185   2019-01-04 13:59:55             0              1
8  60416207185   2019-01-04 21:17:22             0              1
9  60416207185   2019-01-05 00:42:24             0              2


In [48]:
print(
    all_df[
        ["txn_count_1h", "txn_count_24h"]
    ].describe()
)

       txn_count_1h  txn_count_24h
count  1.852394e+06   1.852394e+06
mean   1.999990e-01   4.110729e+00
std    4.756325e-01   3.255181e+00
min    0.000000e+00   0.000000e+00
25%    0.000000e+00   2.000000e+00
50%    0.000000e+00   3.000000e+00
75%    0.000000e+00   6.000000e+00
max    8.000000e+00   3.500000e+01


In [49]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth's radius in km

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

In [50]:
all_df["customer_merchant_distance_km"] = haversine_distance(
    all_df["lat"],
    all_df["long"],
    all_df["merch_lat"],
    all_df["merch_long"]
)

In [51]:
print(
    all_df[
        [
            "lat",
            "long",
            "merch_lat",
            "merch_long",
            "customer_merchant_distance_km"
        ]
    ].head(10)
)

       lat      long  merch_lat  merch_long  customer_merchant_distance_km
0  43.0048 -108.8964  43.974711 -109.741904                     127.606239
1  43.0048 -108.8964  42.018766 -109.044172                     110.308921
2  43.0048 -108.8964  42.961335 -109.157564                      21.787261
3  43.0048 -108.8964  42.228227 -108.747683                      87.204215
4  43.0048 -108.8964  43.321745 -108.091143                      74.212965
5  43.0048 -108.8964  43.477317 -109.467136                      69.984956
6  43.0048 -108.8964  42.871477 -109.160268                      26.099210
7  43.0048 -108.8964  43.332599 -108.318444                      59.375952
8  43.0048 -108.8964  43.598123 -108.977767                      66.302255
9  43.0048 -108.8964  42.314401 -108.554520                      81.700505


In [52]:
print(
    all_df["customer_merchant_distance_km"].describe()
)

count    1.852394e+06
mean     7.611173e+01
std      2.911697e+01
min      2.225452e-02
25%      5.532009e+01
50%      7.821638e+01
75%      9.850947e+01
max      1.521172e+02
Name: customer_merchant_distance_km, dtype: float64


In [53]:
all_df = all_df.sort_values(
    ["merchant", "trans_date_trans_time"]
).reset_index(drop=True)

In [54]:
all_df["merchant_prev_txn_count"] = (
    all_df.groupby("merchant").cumcount()
)

In [56]:
print(
    all_df[
        [
            "merchant",
            "trans_date_trans_time",
            "merchant_prev_txn_count"
        ]
    ].head(20)
)

               merchant trans_date_trans_time  merchant_prev_txn_count
0   fraud_Abbott-Rogahn   2019-01-01 14:43:44                        0
1   fraud_Abbott-Rogahn   2019-01-01 18:09:49                        1
2   fraud_Abbott-Rogahn   2019-01-01 22:52:02                        2
3   fraud_Abbott-Rogahn   2019-01-01 23:26:51                        3
4   fraud_Abbott-Rogahn   2019-01-01 23:33:56                        4
5   fraud_Abbott-Rogahn   2019-01-02 13:30:16                        5
6   fraud_Abbott-Rogahn   2019-01-02 21:27:55                        6
7   fraud_Abbott-Rogahn   2019-01-04 02:07:10                        7
8   fraud_Abbott-Rogahn   2019-01-05 16:17:13                        8
9   fraud_Abbott-Rogahn   2019-01-05 17:31:08                        9
10  fraud_Abbott-Rogahn   2019-01-06 01:53:04                       10
11  fraud_Abbott-Rogahn   2019-01-06 11:41:50                       11
12  fraud_Abbott-Rogahn   2019-01-06 12:22:24                       12
13  fr

In [57]:
print(
    all_df.groupby("dataset")["merchant_prev_txn_count"].describe()
)

             count         mean         std    min     25%     50%     75%  \
dataset                                                                      
test      555719.0  2483.315048  693.938998  727.0  2058.0  2499.0  2952.0   
train    1296675.0  1023.385199  665.733654    0.0   467.0   948.0  1514.0   

            max  
dataset          
test     6261.0  
train    4402.0  


In [58]:
print(
    all_df.groupby("is_fraud")["merchant_prev_txn_count"].describe()
)

              count         mean         std  min    25%     50%     75%  \
is_fraud                                                                   
0         1842743.0  1462.027456  949.922180  0.0  668.0  1355.0  2163.0   
1            9651.0  1334.835457  936.096136  1.0  520.0  1234.0  2007.5   

             max  
is_fraud          
0         6261.0  
1         5808.0  


In [59]:
all_df = all_df.drop(
    columns=["merchant_prev_txn_count"]
)

In [60]:
train_df = all_df[all_df["dataset"] == "train"].copy()
test_df = all_df[all_df["dataset"] == "test"].copy()

In [61]:
train_df = train_df.drop(columns=["dataset"])
test_df = test_df.drop(columns=["dataset"])

In [62]:
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [63]:
feature_cols = [
    "time_since_prev_txn",
    "prev_txn_amt",
    "log_amount_change_ratio",
    "prev_avg_amt",
    "amount_vs_prev_avg",
    "txn_count_1h",
    "txn_count_24h",
    "customer_merchant_distance_km"
]

print(train_df[feature_cols].describe())

       time_since_prev_txn  prev_txn_amt  log_amount_change_ratio  \
count         1.295692e+06  1.295692e+06             1.295692e+06   
mean          5.414169e+02  7.031103e+01             5.186117e-05   
std           7.900163e+02  1.602443e+02             1.868222e+00   
min           0.000000e+00  1.000000e+00            -9.915349e+00   
25%           1.002167e+02  9.650000e+00            -1.136204e+00   
50%           2.760000e+02  4.751000e+01             1.395382e-04   
75%           6.703833e+02  8.312000e+01             1.134704e+00   
max           2.235785e+04  2.894890e+04             9.273822e+00   

       prev_avg_amt  amount_vs_prev_avg  txn_count_1h  txn_count_24h  \
count  1.295692e+06        1.295692e+06  1.296675e+06   1.296675e+06   
mean   7.038964e+01        1.021116e+00  1.895340e-01   3.884149e+00   
std    2.107375e+01        2.657184e+00  4.617663e-01   3.082097e+00   
min    1.030000e+00        2.176430e-03  0.000000e+00   0.000000e+00   
25%    5.802073e+0

In [64]:
print(train_df[feature_cols].isnull().sum())

time_since_prev_txn              983
prev_txn_amt                     983
log_amount_change_ratio          983
prev_avg_amt                     983
amount_vs_prev_avg               983
txn_count_1h                       0
txn_count_24h                      0
customer_merchant_distance_km      0
dtype: int64


In [65]:
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

Train shape: (1296675, 32)
Test shape: (555719, 32)

Train columns:
['trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'gender', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud', 'hour', 'day_of_week', 'month', 'customer_age', 'prev_trans_time', 'time_since_prev_txn', 'prev_txn_amt', 'amount_change_ratio', 'log_amount_change_ratio', 'prev_avg_amt', 'amount_vs_prev_avg', 'txn_count_1h', 'txn_count_24h', 'customer_merchant_distance_km']


In [66]:
print("\nData types:")
print(train_df.dtypes)


Data types:
trans_date_trans_time            datetime64[us]
cc_num                                    int64
merchant                                    str
category                                    str
amt                                     float64
gender                                      str
city                                        str
state                                       str
zip                                       int64
lat                                     float64
long                                    float64
city_pop                                  int64
job                                         str
dob                              datetime64[us]
unix_time                                 int64
merch_lat                               float64
merch_long                              float64
is_fraud                                  int64
hour                                      int32
day_of_week                               int32
month                      

In [67]:
model_features = [
    "merchant",
    "category",
    "amt",
    "gender",
    "city",
    "state",
    "zip",
    "lat",
    "long",
    "city_pop",
    "job",
    "merch_lat",
    "merch_long",
    "hour",
    "day_of_week",
    "month",
    "customer_age",
    "time_since_prev_txn",
    "prev_txn_amt",
    "log_amount_change_ratio",
    "prev_avg_amt",
    "amount_vs_prev_avg",
    "txn_count_1h",
    "txn_count_24h",
    "customer_merchant_distance_km"
]

In [68]:
X_train = train_df[model_features].copy()
y_train = train_df["is_fraud"].copy()

X_test = test_df[model_features].copy()
y_test = test_df["is_fraud"].copy()

In [69]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (1296675, 25)
y_train: (1296675,)
X_test: (555719, 25)
y_test: (555719,)


In [70]:
import catboost
print(catboost.__version__)

ModuleNotFoundError: No module named 'catboost'

In [71]:
train_df.to_parquet("processed_train.parquet", index=False)
test_df.to_parquet("processed_test.parquet", index=False)